In [ ]:
加载预训练模型和数据集:使用 transformers 库加载预训练的模型和 GSM8K 数据集。

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# 加载预训练模型和tokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 加载GSM8K数据集
load_dataset('gsm8k', 'main')

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 194.31it/s, Materializing param=model.norm.weight]                              


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [ ]:
数据预处理:将问题和答案转换为模型可理解的输入格式（tokens）。
使用tokenizer对问题和答案进行编码。

In [ ]:
def preprocess_data(examples):
    inputs = [ex["question"] for ex in examples["train"]]
    outputs = [ex["answer"] for ex in examples["train"]]

    # 使用 tokenizer 对数据进行编码
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    model_outputs = tokenizer(outputs, padding="max_length", truncation=True, max_length=128)

    # 返回最终的编码数据
    model_inputs["labels"] = model_outputs["input_ids"]
    return model_inputs

# 进行预处理
train_dataset = dataset["train"].map(preprocess_data, batched=True)

In [ ]:
定义训练参数   

定义训练过程中需要的参数，包括学习率、batch size、训练轮次等。使用 Trainer API 来简化这个过程。

In [ ]:
from transformers import Trainer, TrainingArguments

# 设置训练参数
training_args = TrainingArguments(
    output_dir="./results",          # 输出目录
    evaluation_strategy="epoch",     # 每个epoch后进行评估
    learning_rate=2e-5,              # 学习率
    per_device_train_batch_size=4,   # 每设备的batch size
    per_device_eval_batch_size=8,    # 每设备的eval batch size
    num_train_epochs=3,              # 训练的epoch数
    weight_decay=0.01,               # 权重衰减
    logging_dir='./logs',            # 日志文件夹
)

# 创建Trainer对象
trainer = Trainer(
    model=model,                       # 模型
    args=training_args,                # 训练参数
    train_dataset=train_dataset,       # 训练数据集
    eval_dataset=dataset["test"],      # 测试数据集
)

In [ ]:
开始训练

In [ ]:
trainer.train()

In [ ]:
评估模型

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
模型保存

In [ ]:
model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")

In [ ]:
进行推理

最后，使用微调后的模型进行推理，并查看模型对新问题的回答。

In [ ]:
# 加载微调后的模型
model = AutoModelForCausalLM.from_pretrained("./finetuned_model")
tokenizer = AutoTokenizer.from_pretrained("./finetuned_model")

# 输入新的问题
question = "What is 45 + 67?"
inputs = tokenizer(question, return_tensors="pt")

# 生成模型的输出
outputs = model.generate(**inputs, max_new_tokens=50)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)